# Practicing NPL estimation using Zurcher model

Class 14 of ECO 629, notes at <https://dse.iskh.me/npl>.

The same model estimated three ways on Harold Zurcher's data and on panels simulated from the model, where the true parameters are known:

- **NFXP**, the maximum likelihood benchmark of [Class 10](https://dse.iskh.me/nfxp)
- **two-step CCP** with the one-period finite dependence of [Class 13](https://dse.iskh.me/ccp-practice)
- **NPL**, the iterated pseudo-maximum likelihood estimator of this class, built on the policy iteration operator $\Psi(P,\theta)$

In [ ]:
import sys
sys.path.insert(0, '../session10-11')        # zurcher.py, nfxp.py and the bus data
sys.path.insert(0, '../session13-oct6')      # sim_zurcher.py

import time
import numpy as np
import matplotlib.pyplot as plt
import nfxp                                  # read_busdata, estim_zurcher
import ccp                                   # flexible_logit, ccp_second_stage
from sim_zurcher import simulate             # simulator for zurcher model

t0 = time.perf_counter()
def say(msg):
    '''Progress line with the time elapsed since t0, reset at the start of each long cell'''
    print(f'[{time.perf_counter() - t0:7.1f} s] {msg}', flush=True)   # flush: show it while the cell runs

# Original data
say('reading the bus data')
data = nfxp.read_busdata()                   # bus groups 1-4, 175 bins, 450 000 miles
m0 = nfxp.estim_zurcher(data)                # the model with the data attached
say(f'bus data: NFXP full estimation on {m0.N} bus-months')
m0_res = m0.estimate(full=True)              # NFXP full estimation
say('bus data: solving the model at the NFXP estimates')
ev0, pk0, _ = m0.solve()                     # the model at the NFXP estimates
print(m0_res)

# Simulated data
Ns = (10_000, 50_000)                        # buses, 120 months each
m, m_res, ev, pk = [], [], [], []            # one entry per simulated sample, in the order of Ns
for N in Ns:
    say(f'{N} buses: simulating 120 months from the model at the bus-data estimates')
    sim = simulate(m0, N=N, T=120, seed=1, pk=pk0, x0=0)    # new engines at the start, like the data
    m.append(nfxp.estim_zurcher(sim, J=m0.J, p=m0.p))       # same grid and transition probabilities
    say(f'{N} buses: NFXP full estimation on {m[-1].N} bus-months')
    m_res.append(m[-1].estimate(full=True))                 # NFXP full estimation
    say(f'{N} buses: solving the model at the NFXP estimates')
    ev_i, pk_i, _ = m[-1].solve()                           # the model at the NFXP estimates
    ev.append(ev_i)
    pk.append(pk_i)
say('done')

## 2-step CCP

The two-step estimator of [Class 13](https://dse.iskh.me/ccp-practice), with `flexible_logit()` and `ccp_second_stage()` from `ccp.py`: a Chebyshev logit of degree 3 and 5 in the first stage, and in the second the static logit with the offset from the [one-period finite dependence](https://dse.iskh.me/ccp-practice#zurcher-finite-dependence) of the replacement action. The NFXP estimates of the cell above, on the same sample, are the benchmark.

First on Zurcher's data, then on the panels simulated from the model at the NFXP estimates. The simulated panels start every bus with a new engine, like the data, and estimate the transition probabilities from their own transitions.

The finite dependence offset uses the CCPs one period ahead only, not the full policy valuation $\varphi(\hat P)$. Watch the standard errors of $c$: this representation does not have the zero Jacobian property, so the estimator is [not asymptotically equivalent to the MLE](https://dse.iskh.me/npl#npl-asymptotics).

In [ ]:
def ccp_estimates(model, degrees=(3, 5)):
    '''Two-step CCP estimates for each first-stage degree: {degree: (theta, se, loglik)},
    None where the first stage underflows to P(replace|x) = 0 and the offset is -inf'''
    out = {}
    for k in degrees:
        say(f'  two-step CCP, Chebyshev degree {k}')
        with np.errstate(divide='ignore', invalid='ignore'):
            try:
                say('    first stage: flexible logit')
                first_stage = ccp.flexible_logit(model, k)
                say('    second stage: logit with offset')
                out[k] = ccp.ccp_second_stage(model, first_stage)
            except (np.linalg.LinAlgError, RuntimeError):
                say(f'    failed, P(replace|x) = 0 somewhere')
                out[k] = None
    return out

t0 = time.perf_counter()                     # restart the clock of say()
say('Original bus data')
m0_ccp = ccp_estimates(m0)                   # two-step CCP on the bus data
m_ccp = []                                   # one entry per simulated sample, in the order of Ns
for N, mi in zip(Ns, m):
    say(f'Simulated sample with {N} buses, {mi.N} bus-months')
    m_ccp.append(ccp_estimates(mi))          # two-step CCP on the same sample as NFXP
say('done')

In [ ]:
# RC and c from every estimator, standard errors in brackets;
# the simulated samples are drawn at the NFXP estimates of m0
def est_se(theta, se):
    '''RC and c as "estimate (s.e.)" columns'''
    return ''.join(f'{f"{t:.3f} ({s:.3f})":>18}' for t, s in zip(theta[:2], se[:2]))

for name, r, c in zip(['Zurcher bus data     '] + [f'{N} simulated buses' for N in Ns], [m0_res] + m_res, [m0_ccp] + m_ccp):
    print(f'\n{name:<16}{"RC":>13}{"c":>18}')
    print(f'{"NFXP":<16}' + est_se(r.theta, r.se))
    for k, e in c.items():
        print(f'{"CCP, degree " + str(k):<16}' + (est_se(e[0], e[1]) if e else f'{"fails":>18}'))

## NPL with the policy iteration operator

[Policy valuation](https://dse.iskh.me/npl#from-value-space-to-probability-space) at given CCPs $P$:

$$
V^\sigma = \varphi(P) = [I - \beta \Pi]^{-1} \sum_{d} P(d) \ast \big[u(d) + e(d)\big],
\qquad \Pi = \sum_{d} P(d) \ast \Pi(d),
$$

with $\Pi(\text{keep})$ the transition matrix `trpr` of the model, $\Pi(\text{replace})$ its first row (a fresh engine) repeated for every $x$, and the EV1 correction term $e(x,d) = \gamma - \log P(d|x)$. The code drops $\gamma$, as the Bellman operator of `zurcher.py` does: it shifts $V^\sigma$ by a constant, which cancels from all value differences.

The utility of the bus model is linear in $\theta = (RC, c)$, $u(x,d) = h(x,d)'\theta$, so $V^\sigma$ is linear in $\theta$ too, and the value difference that enters the logit is

$$
v(x,\text{keep}) - v(x,\text{replace}) = \tilde h(x)'\theta + \tilde e(x),
$$

with $\tilde h$ and $\tilde e$ computed by `npl_regressors()` from $P$ alone. The code carries the CCPs as `pk` $= P(\text{keep}|x)$. Then:

- the policy iteration operator is $\Psi(P,\theta)(\text{keep}|x) = \Lambda\big(\tilde h(x)'\theta + \tilde e(x)\big)$, function `psi()`
- the [pseudo-likelihood](https://dse.iskh.me/npl#pseudo-maximum-likelihood-estimation) $\ell_n(\theta) = \sum \log \Psi(\hat P,\theta)(d_{it}|x_{it})$ is a static logit with regressors $\tilde h(x)$ and offset $\tilde e(x)$, globally concave in $\theta$, maximized by the same `ccp.logit_mle` as the two-step CCP estimator
- the [NPL iterations](https://dse.iskh.me/npl#swapping-the-nested-fixed-point) alternate the pseudo-likelihood step $\hat\theta_K$ and the policy iteration step $\hat P_K = \Psi(\hat P_{K-1}, \hat\theta_K)$, function `npl_estimate()`, which returns the fixed point

Step $K = 1$ is the pseudo-maximum likelihood estimator, and stopping after $K$ steps gives the [$K$-stage policy iteration estimators](https://dse.iskh.me/npl#the-sequence-of-estimators). By the [zero Jacobian property](https://dse.iskh.me/npl#npl-zero-jacobian) the fixed point of the iterations is the NFXP estimate. The $K$-stage estimates along the way are passed to the `callback`, which the cells below use to print and keep them.

In [ ]:
def npl_regressors(model, pk):
    '''Policy valuation at the CCPs pk = P(keep|x): regressors h (n x 2) and offset e (n,)
    such that v(x,keep) - v(x,replace) = h[x] @ (RC, c) + e[x]'''
    n, beta = model.n, model.beta
    Fk = model.trpr                                    # transitions after keep: row x
    Fr = np.tile(model.trpr[0], (n, 1))                # after replace: a fresh engine, row 0 for every x
    FU = Fr + pk[:, None] * (Fk - Fr)                  # unconditional transitions under pk
    Hk = np.column_stack([np.zeros(n), -0.001 * model.grid])   # u(x,keep)    = Hk @ (RC, c)
    Hr = np.column_stack([-np.ones(n), np.zeros(n)])           # u(x,replace) = Hr @ (RC, c)
    ent = -pk * np.log(pk) - (1 - pk) * np.log(1 - pk)   # sum_d P(d) e(d) with e(d) = -log P(d), gamma dropped
    # More precise when P(replace|x) is tiny: work with the log odds instead of 1 - pk
    # z = np.log(pk) - np.log1p(-pk)                     # log odds of keeping
    # ent = ccp.logsum_binary(z) - pk * z                # the same sum_d P(d) e(d)
    rhs = np.column_stack([Hr + pk[:, None] * (Hk - Hr), ent])   # expected flow utility and correction term
    W = np.linalg.solve(np.eye(n) - beta * FU, rhs)    # policy valuation: V = W[:, :2] @ theta + W[:, 2]
    Z = np.column_stack([Hk - Hr, np.zeros(n)]) + beta * (Fk - Fr) @ W   # v(keep) - v(replace), same split
    return Z[:, :2], Z[:, 2]

def psi(model, pk, theta):
    '''Policy iteration operator: P(keep|x) after one policy valuation at pk and improvement at theta'''
    h, e = npl_regressors(model, pk)
    return ccp.logit_binary(h @ theta + e)

def npl_estimate(model, pk, tol=1e-8, maxiter=100, callback=None):
    '''NPL estimator started from the first-stage CCPs pk = P(keep|x),
    with given tolerance and number of iterations.
    Callback function is invoked at each iteration if given, with the K-stage estimates:
    callback(K=K, theta=theta, se=se, loglik=loglik, pk=pk, err=err).
    Returns theta, standard errors, pseudo-log-likelihood and the CCPs at the NPL fixed point.'''
    keep = (model.d == 0).astype(float)                # y = 1 for keep in the pseudo-likelihood
    for k in range(maxiter):
        h, e = npl_regressors(model, pk)                                     # one linear solve
        theta, se, ll = ccp.logit_mle(h[model.x], keep, e[model.x])         # pseudo-likelihood step
        pk1 = ccp.logit_binary(h @ theta + e)                                # policy iteration: psi(model, pk, theta)
        err = np.abs(pk1 - pk).max()
        if callback is not None: 
            callback(K=k+1, theta=theta, se=se, loglik=ll, pk=pk1, err=err)
        if err < tol: break
        pk = pk1
    else:
        raise RuntimeError('Failed to converge in %d iterations' % maxiter)
    return theta, se, ll, pk1

In [ ]:
t0 = time.perf_counter()                     # restart the clock of say()
pk_first = ccp.flexible_logit(m0, 3, keep=True)   # first stage: P(keep|x), Chebyshev logit of degree 3

kstage = {}                                  # K-stage PI estimates, filled in by the callback
def print_kstage(K, theta, se, loglik, err, **kwargs):
    '''Callback: print the K-stage estimates and keep them'''
    kstage[K] = (theta, se)
    print(f'{K:5d}' + est_se(theta, se) + f'{loglik:11.2f}{err:11.2e}')

def print_ccp(c):
    '''Rows of the two-step CCP estimates computed above, one per first-stage degree'''
    for k, e in c.items():
        print(f'{"CCP, degree " + str(k):<20}' + (est_se(e[0], e[1]) if e else f'{"fails":>18}'))

say('bus data: NPL step by step')
print(f'{"K":>5}{"RC":>18}{"c":>18}{"loglik":>11}{"err":>11}')
theta, se, ll, pk_npl = npl_estimate(m0, pk_first, callback=print_kstage)
say('done')

K = max(kstage)                              # the number of steps to convergence
print(f'\n{"Zurcher bus data":<20}{"RC":>18}{"c":>18}')
print(f'{"NFXP":<20}' + est_se(m0_res.theta, m0_res.se))
print_ccp(m0_ccp)
print(f'{"NPL, K = 1":<20}' + est_se(*kstage[1]))
print(f'{"NPL, K = " + str(K):<20}' + est_se(theta, se))

In [ ]:
t0 = time.perf_counter()                     # restart the clock of say()

def keep_kstage(K, theta, se, **kwargs):
    '''Callback: keep the K-stage estimates, print nothing'''
    kstage[K] = (theta, se)

for N, mi, ri, ci in zip(Ns, m, m_res, m_ccp):
    say(f'Simulated sample with {N} buses: first stage, then NPL')
    kstage = {}
    theta, se, ll, _ = npl_estimate(mi, ccp.flexible_logit(mi, 3, keep=True), callback=keep_kstage)
    K = max(kstage)
    say(f'  converged in K = {K} steps')
    print(f'{f"{N} simulated buses":<22}{"RC":>16}{"c":>18}')
    print(f'{"true":<20}{m0_res.theta[0]:18.3f}{m0_res.theta[1]:18.3f}')
    print(f'{"NFXP":<20}' + est_se(ri.theta, ri.se))
    print_ccp(ci)
    print(f'{"NPL, K = 1":<20}' + est_se(*kstage[1]))
    print(f'{"NPL, K = " + str(K):<20}' + est_se(theta, se))
say('done')